# 0. Preparation

In [1]:
import os
import sys
import random
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import make_scorer, mean_absolute_percentage_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_percentage_error
from IPython.display import display

import warnings
warnings.filterwarnings('ignore')

# Detect environment
def detect_environment() -> str:
    """Detect if running in Colab, Kaggle, or local environment"""
    if 'google.colab' in sys.modules:
        return 'colab'
    elif 'kaggle_secrets' in sys.modules or os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return 'kaggle'
    else:
        return 'local'

ENV = detect_environment()
print(f"🔍 Detected environment: {ENV.upper()}")

🔍 Detected environment: KAGGLE


# 1. Dataset Download and Setup

In [2]:
def setup_dataset_paths(env: str)  -> dict[str, str]:
    if env == 'colab':
        dataset_name = "intelecta-cup-data-mining-competition"
        local_path = "/content/dataset"

        if not os.path.exists(local_path):
            print("📥 Setting up tabular dataset in Colab...")

            kaggle_json_drive = "/content/drive/MyDrive/kaggle.json"
            kaggle_json_local = "/root/.kaggle/kaggle.json"

            if os.path.exists(kaggle_json_drive):
                print("📂 Copying kaggle.json from Drive...")
                !mkdir -p /root/.kaggle
                !cp "{kaggle_json_drive}" "{kaggle_json_local}"
                !chmod 600 "{kaggle_json_local}"
                print("✅ Kaggle credentials loaded from Drive")
            else:
                print("⚠️ Please upload kaggle.json manually or place it in Drive root folder")
                from google.colab import files
                uploaded = files.upload()
                for fn in uploaded.keys():
                    !mkdir -p /root/.kaggle
                    !mv "{fn}" "/root/.kaggle/kaggle.json"
                    !chmod 600 "/root/.kaggle/kaggle.json"
                    print(f"✅ Kaggle credentials uploaded: {fn}")

            # --- Download dataset ---
            !kaggle competitions download -c {dataset_name} -p /content

            import zipfile
            zip_path = f"/content/{dataset_name}.zip"
            if os.path.exists(zip_path):
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(local_path)
                print(f"✅ Dataset extracted to {local_path}")
            else:
                raise FileNotFoundError(f"Dataset zip not found: {zip_path}")

        return {
            'train_csv': f"{local_path}/train.csv",
            'test_csv': f"{local_path}/test.csv",
            'save_dir': "/content/drive/MyDrive/PROJECTS/Cognivio/models"
        }

    elif env == 'kaggle':
        return {
            'train_csv': "/kaggle/input/intelecta-cup-data-mining-competition/train.csv",
            'test_csv': "/kaggle/input/intelecta-cup-data-mining-competition/test.csv",
            'save_dir': "/kaggle/working"
        }

    else:
        base_path = "data"
        return {
            'train_csv': f"{base_path}/train.csv",
            'test_csv': f"{base_path}/test.csv",
            'save_dir': "models"
        }

# Setup paths
paths = setup_dataset_paths(ENV)
print(f"📁 Dataset paths configured for {ENV}:")
for key, path in paths.items():
    exists = "✅" if os.path.exists(path) else "⚠️"
    print(f"   {key}: {path} {exists}")

# Create save directory
os.makedirs(paths['save_dir'], exist_ok=True)

# Set random seed for reproducibility
def set_seed(seed: int = 20) -> None:
    random.seed(seed)
    np.random.seed(seed)

📁 Dataset paths configured for kaggle:
   train_csv: /kaggle/input/intelecta-cup-data-mining-competition/train.csv ✅
   test_csv: /kaggle/input/intelecta-cup-data-mining-competition/test.csv ✅
   save_dir: /kaggle/working ✅


# 3. Dataset Overview

In [3]:
df = pd.read_csv(paths['train_csv'])

display(df.head(20))
display(df.describe())
display(df.info())

,ID,Tahun,Nama_Negara,Wilayah,Jenis_Tanaman,Total_Curah_Hujan_mm,Emisi_CO2_JT_Ton,Hasil_Panen_Ton_per_HA,Kejadian_Cuaca_Ekstrim,Akses_Irigasi,Penggunaan_Pestisida_KG_per_HA,Penggunaan_Pupuk_KG_per_HA,Indeks_Kesehatan_Tanah,Strategi_Adaptasi,Suhu_Rata_Rata_C
0,0,2015,USA,South,Soybeans,1658.71,13.36,2.620,10,74.41,38.97,2.64,46.07,Manajemen Air,20.43
1,1,2022,China,East,Wheat,1478.74,9.55,0.570,2,36.90,49.99,77.22,88.87,Rotasi Tanaman,-0.33
2,2,2000,India,'West Bengal',Fruits,1252.34,27.37,2.115,3,34.21,2.75,83.94,77.15,Pertanian Organik,12.97
3,3,2008,Nigeria,'North West',Sugarcane,209.89,16.16,4.158,5,91.74,36.80,37.50,73.59,Pertanian Organik,12.81
4,4,1991,Canada,Ontario,Vegetables,1086.67,3.71,2.430,0,14.72,7.22,28.72,41.90,Tanpa Adaptasi,4.22
5,5,1990,Australia,Queensland,Fruits,2265.29,2.33,2.030,4,79.39,48.80,3.30,77.63,Tanpa Adaptasi,14.70
6,6,2017,Russia,Northwestern,Rice,586.35,7.00,1.070,10,88.96,42.20,71.15,45.82,Tanaman Tahan Kekeringan,7.34
7,7,1995,China,Central,Vegetables,2281.25,24.21,2.907,1,27.44,23.94,13.96,50.97,Tanaman Tahan Kekeringan,16.97
8,8,2016,France,'Provence-Alpes-Cote d’Azur',NaN,792.07,10.31,3.420,1,51.51,4.60,75.99,94.94,Pertanian Organik,30.23
9,9,1991,Nigeria,'North West',Cotton,1746.26,25.28,2.772,7,39.09,9.33,90.96,59.65,Manajemen Air,20.30


,ID,Tahun,Total_Curah_Hujan_mm,Emisi_CO2_JT_Ton,Hasil_Panen_Ton_per_HA,Kejadian_Cuaca_Ekstrim,Akses_Irigasi,Penggunaan_Pestisida_KG_per_HA,Penggunaan_Pupuk_KG_per_HA,Indeks_Kesehatan_Tanah,Suhu_Rata_Rata_C
count,8000.00000,8000.000000,7821.000000,8000.000000,7800.000000,8000.000000,7819.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,3999.50000,2007.032750,1615.503060,15.271184,2.238219,4.989750,55.394575,24.920015,49.706654,64.824446,15.206680
std,2309.54541,10.106035,807.932322,8.551214,0.996626,3.171814,26.034847,14.454507,28.674985,20.153617,11.490611
min,0.00000,1990.000000,200.170000,0.500000,0.450000,0.000000,10.010000,0.000000,0.030000,30.000000,-4.990000
25%,1999.75000,1998.000000,929.290000,7.860000,1.449000,2.000000,32.905000,12.550000,25.160000,47.115000,5.377500
50%,3999.50000,2007.000000,1614.790000,15.250000,2.170000,5.000000,55.340000,24.930000,49.270000,64.675000,15.140000
75%,5999.25000,2016.000000,2316.820000,22.820000,2.930000,8.000000,77.770000,37.382500,74.430000,82.302500,25.340000
max,7999.00000,2024.000000,2999.670000,30.000000,5.000000,10.000000,99.990000,49.990000,99.990000,100.000000,35.000000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   ID                              8000 non-null   int64  
 1   Tahun                           8000 non-null   int64  
 2   Nama_Negara                     8000 non-null   object 
 3   Wilayah                         8000 non-null   object 
 4   Jenis_Tanaman                   7781 non-null   object 
 5   Total_Curah_Hujan_mm            7821 non-null   float64
 6   Emisi_CO2_JT_Ton                8000 non-null   float64
 7   Hasil_Panen_Ton_per_HA          7800 non-null   float64
 8   Kejadian_Cuaca_Ekstrim          8000 non-null   int64  
 9   Akses_Irigasi                   7819 non-null   float64
 10  Penggunaan_Pestisida_KG_per_HA  8000 non-null   float64
 11  Penggunaan_Pupuk_KG_per_HA      8000 non-null   float64
 12  Indeks_Kesehatan_Tanah          80

None

In [4]:
# Check for issues
print("🔍 Data Diagnostics:")
print(f"Target range: {df['Suhu_Rata_Rata_C'].min():.2f} to {df['Suhu_Rata_Rata_C'].max():.2f}")
print(f"Target mean: {df['Suhu_Rata_Rata_C'].mean():.2f}")
print(f"Target std: {df['Suhu_Rata_Rata_C'].std():.2f}")
print(f"\nZero values in target: {(df['Suhu_Rata_Rata_C'] == 0).sum()}")
print(f"Negative values in target: {(df['Suhu_Rata_Rata_C'] < 0).sum()}")
print(f"Missing values in target: {df['Suhu_Rata_Rata_C'].isnull().sum()}")

🔍 Data Diagnostics:
Target range: -4.99 to 35.00
Target mean: 15.21
Target std: 11.49

Zero values in target: 0
Negative values in target: 971
Missing values in target: 0


# 2. Data Preprocessing

In [5]:
# Handle missing values
numeric_features = ['Total_Curah_Hujan_mm', 'Hasil_Panen_Ton_per_HA', 'Akses_Irigasi']
categorical_features = ['Jenis_Tanaman', 'Nama_Negara', 'Wilayah', 'Strategi_Adaptasi']

# Impute numeric features with median
numeric_imputer = SimpleImputer(strategy='median')
df[numeric_features] = numeric_imputer.fit_transform(df[numeric_features])

# Impute categorical with mode
cat_imputer = SimpleImputer(strategy='most_frequent')
df[categorical_features] = cat_imputer.fit_transform(df[categorical_features])

# Encode categorical variables
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    df[f'{col}_encoded'] = le.fit_transform(df[col])
    label_encoders[col] = le

# Feature engineering
df['CO2_per_Rainfall'] = df['Emisi_CO2_JT_Ton'] / (df['Total_Curah_Hujan_mm'] + 1)
df['Pesticide_per_Fertilizer'] = df['Penggunaan_Pestisida_KG_per_HA'] / (df['Penggunaan_Pupuk_KG_per_HA'] + 1)
df['Yield_Efficiency'] = df['Hasil_Panen_Ton_per_HA'] * df['Indeks_Kesehatan_Tanah'] / 100

# Prepare features and target
feature_cols = [col for col in df.columns if col not in ['Suhu_Rata_Rata_C', 'ID'] + categorical_features]
X = df[feature_cols]
y = df['Suhu_Rata_Rata_C']

# Split data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# 4. Model Selection

In [6]:
def mape_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(mean_absolute_percentage_error(y_true, y_pred) * 100)

# Model 1: XGBoost (handles non-linearity well)
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:absoluteerror'  # Better for MAPE
)

# Model 2: LightGBM (fast and accurate)
lgbm_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='mae'  # Mean Absolute Error objective
)

# Model 3: Gradient Boosting
gb_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    random_state=42
)

# Ensemble model
ensemble_model = VotingRegressor([
    ('xgb', xgb_model),
    ('lgbm', lgbm_model),
    ('gb', gb_model)
])

# Train ensemble
print("🔧 Training ensemble model...")
ensemble_model.fit(X_train_scaled, y_train)

# Evaluate
y_pred_train = ensemble_model.predict(X_train_scaled)
y_pred_val = ensemble_model.predict(X_val_scaled)

train_mape = mape_score(y_train, y_pred_train)
val_mape = mape_score(y_val, y_pred_val)

print(f"📊 Training MAPE: {train_mape:.4f}%")
print(f"📊 Validation MAPE: {val_mape:.4f}%")

🔧 Training ensemble model...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001280 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2661
[LightGBM] [Info] Number of data points in the train set: 6400, number of used features: 16
[LightGBM] [Info] Start training from score 15.260000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furth

In [7]:
# Check predictions distribution
print(f"\n📊 Validation Predictions:")
print(f"Pred range: {y_pred_val.min():.2f} to {y_pred_val.max():.2f}")
print(f"Pred mean: {y_pred_val.mean():.2f}")
print(f"Actual range: {y_val.min():.2f} to {y_val.max():.2f}")
print(f"Actual mean: {y_val.mean():.2f}")

# Check for near-zero predictions (causes huge MAPE)
near_zero_mask = np.abs(y_pred_val) < 0.1
print(f"\n⚠️ Predictions near zero: {near_zero_mask.sum()} / {len(y_pred_val)}")
if near_zero_mask.sum() > 0:
    print(f"Actual values for near-zero predictions: {y_val[near_zero_mask].values}")


📊 Validation Predictions:
Pred range: -1.29 to 27.97
Pred mean: 14.73
Actual range: -4.98 to 34.98
Actual mean: 14.66

⚠️ Predictions near zero: 1 / 1600
Actual values for near-zero predictions: [-2.11]


# 5. Hyperparameter Tuning

In [8]:
# Define parameter grid for XGBoost
param_grid = {
    'n_estimators': [300, 500, 700],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [5, 7, 9],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
}

mape_scorer = make_scorer(lambda y_true, y_pred: -mape_score(y_true, y_pred))

# Randomized search
random_search = RandomizedSearchCV(
    XGBRegressor(random_state=42, objective='reg:absoluteerror'),
    param_distributions=param_grid,
    n_iter=20,
    scoring=mape_scorer,
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

print("🔍 Hyperparameter tuning...")
random_search.fit(X_train_scaled, y_train)

print(f"✅ Best parameters: {random_search.best_params_}")
print(f"✅ Best CV MAPE: {-random_search.best_score_:.4f}%")

best_model = random_search.best_estimator_

🔍 Hyperparameter tuning...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
✅ Best parameters: {'subsample': 0.9, 'n_estimators': 700, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.9}
✅ Best CV MAPE: 376.1783%


# 6. Generate Predictions

In [9]:
# Load test data
df_test = pd.read_csv(paths['test_csv'])

# Apply same preprocessing
df_test[numeric_features] = numeric_imputer.transform(df_test[numeric_features])
df_test[categorical_features] = cat_imputer.transform(df_test[categorical_features])

for col in categorical_features:
    df_test[f'{col}_encoded'] = label_encoders[col].transform(df_test[col])

# Feature engineering on test
df_test['CO2_per_Rainfall'] = df_test['Emisi_CO2_JT_Ton'] / (df_test['Total_Curah_Hujan_mm'] + 1)
df_test['Pesticide_per_Fertilizer'] = df_test['Penggunaan_Pestisida_KG_per_HA'] / (df_test['Penggunaan_Pupuk_KG_per_HA'] + 1)
df_test['Yield_Efficiency'] = df_test['Hasil_Panen_Ton_per_HA'] * df_test['Indeks_Kesehatan_Tanah'] / 100

X_test = df_test[feature_cols]
X_test_scaled = scaler.transform(X_test)

# Predict
predictions = best_model.predict(X_test_scaled)

# Save submission
submission = pd.DataFrame({
    'ID': df_test['ID'],
    'Suhu_Rata_Rata_C': predictions
})
submission.to_csv(f"{paths['save_dir']}/submission.csv", index=False)
print(f"✅ Submission saved to {paths['save_dir']}/submission.csv")

✅ Submission saved to /kaggle/working/submission.csv
